# 🍅 AgriTwin-GH — Tomato Leaf Disease Classifier

---

## Quick-Start Instructions

### 1. Dataset Location
**Local / VS Code:** Place (or ensure) the dataset folders at the paths below relative to your repo root:
```
data/external/Tomato Diseases/          ← subfolder per disease class
    Tomato_Early_Blight/
    Tomato_Late_Blight/
    Tomato_Leaf_Mold/
    Tomato_Powdery_Mildew/
    Tomato_Spider_Mites/
data/external/Tomato Healthy Leaves/    ← images directly inside (flat)
```
**Google Colab:**
```python
# Option A — Google Drive mount
from google.colab import drive
drive.mount('/content/drive')
# Then set CONFIG['repo_root'] = '/content/drive/MyDrive/AgriTwin-GH'

# Option B — Upload zip and extract
# !unzip /content/dataset.zip -d /content/AgriTwin-GH
# Then set CONFIG['repo_root'] = '/content/AgriTwin-GH'
```

### 2. CONFIG Values to Adjust
| Key | Default | When to Change |
|-----|---------|----------------|
| `repo_root` | `'.'` | Set full path in Colab |
| `backbone_name` | `'EfficientNetB0'` | Try B3, ResNet50, MobileNetV3, DenseNet121 |
| `batch_size` | `32` | Lower to 16 if GPU OOM |
| `epochs_warmup` | `10` | Fewer for quick tests |
| `epochs_finetune` | `20` | Main training budget |
| `loss_type` | `'ce'` | Switch to `'focal'` for heavy imbalance |
| `mixed_precision` | `True` | Set False if numerical issues |

### 3. How to Run
Run all cells top-to-bottom: **Runtime → Run all** (Colab) or **Run All** (VS Code).
The two training phases (warm-up → fine-tune) are sequential — do not skip.

### 4. Output Locations
```
src/agritwin_gh/models/<run_id>.keras          ← saved model
src/agritwin_gh/models/artifacts/<run_id>/
    label_map.json
    metrics.json
    confusion_matrix.png
    misclassified_grid.png
    training_history.csv
```
---

## Section A — Environment Setup

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — All tunable parameters in one place.
# Modify only this cell before running the notebook.
# ─────────────────────────────────────────────────────────────────────────────
import datetime

# Auto-generate a unique run ID (timestamp-based)
_RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    # ── Repo / Dataset Paths ──────────────────────────────────────────────────
    # In Colab, change repo_root to the full path after mounting Drive.
    # e.g. '/content/drive/MyDrive/AgriTwin-GH' or '/content/AgriTwin-GH'
    "repo_root"         : ".",
    "disease_folder"    : "data/external/Tomato Diseases",
    "healthy_folder"    : "data/external/Tomato Healthy Leaves",

    # Disease sub-folders to INCLUDE (Tomato_Septoria_Leaf_Spot is excluded)
    "include_disease_folders": [
        "Tomato_Early_Blight",
        "Tomato_Late_Blight",
        "Tomato_Leaf_Mold",
        "Tomato_Powdery_Mildew",
        "Tomato_Spider_Mites",
    ],

    # Folder-name → canonical label mapping
    "folder_to_label": {
        "Tomato_Early_Blight"   : "tomato_early_blight",
        "Tomato_Late_Blight"    : "tomato_late_blight",
        "Tomato_Leaf_Mold"      : "tomato_leaf_mold",
        "Tomato_Powdery_Mildew" : "tomato_powdery_mildew",
        "Tomato_Spider_Mites"   : "tomato_spider_mites",
        "Tomato Healthy Leaves" : "tomato_leaf_healthy",
    },

    # ── Image / Batch ─────────────────────────────────────────────────────────
    "image_size"        : (224, 224),   # (H, W) — EfficientNetB0 native input
    "batch_size"        : 32,
    "num_channels"      : 3,

    # ── Split Ratios (used when no pre-split folders exist) ───────────────────
    "val_split"         : 0.15,         # fraction of total data for validation
    "test_split"        : 0.10,         # fraction of total data for test
    "random_seed"       : 42,

    # ── Backbone ──────────────────────────────────────────────────────────────
    # Supported: 'EfficientNetB0', 'EfficientNetB3', 'ResNet50',
    #            'MobileNetV3Large', 'DenseNet121'
    "backbone_name"     : "EfficientNetB0",

    # ── Training Schedule ─────────────────────────────────────────────────────
    "epochs_warmup"     : 10,           # Phase 1: backbone frozen
    "epochs_finetune"   : 25,           # Phase 2: top backbone layers unfrozen
    "unfreeze_top_layers" : 30,         # how many top backbone layers to unfreeze

    # ── Learning Rates ────────────────────────────────────────────────────────
    "lr_warmup"         : 1e-3,
    "lr_finetune"       : 5e-5,

    # ── Custom Head ──────────────────────────────────────────────────────────
    "head_dropout_1"    : 0.4,
    "head_units"        : 256,
    "head_dropout_2"    : 0.3,
    "num_classes"       : 6,

    # ── Loss ──────────────────────────────────────────────────────────────────
    # 'ce'    → CategoricalCrossentropy with label_smoothing
    # 'focal' → Sigmoid Focal Cross-Entropy (via tf.keras or custom)
    "loss_type"         : "ce",
    "label_smoothing"   : 0.1,
    # Focal loss params (only used when loss_type == 'focal')
    "focal_alpha"       : 0.25,
    "focal_gamma"       : 2.0,

    # ── Augmentation Strength ────────────────────────────────────────────────
    "aug_rotation_factor"   : 0.15,    # ±15% rotation
    "aug_zoom_factor"       : 0.15,    # ±15% zoom
    "aug_brightness_delta"  : 0.15,    # brightness shift range
    "aug_contrast_factor"   : 0.15,    # [1-f, 1+f] contrast multiply
    "aug_hue_delta"         : 0.05,    # hue shift (small, keep leaf colours)
    "aug_saturation_lower"  : 0.8,
    "aug_saturation_upper"  : 1.2,
    "aug_crop_fraction"     : 0.90,    # random crop retains ≥90% of image
    "aug_cutout_fraction"   : 0.15,    # cutout patch is 15% of image dimension

    # ── Mixed Precision ───────────────────────────────────────────────────────
    "mixed_precision"   : True,        # set False if you see NaN losses

    # ── Output Paths ──────────────────────────────────────────────────────────
    "models_dir"        : "src/agritwin_gh/models",
    "artifacts_dir"     : "src/agritwin_gh/models/artifacts",
    "run_id"            : _RUN_TIMESTAMP,

    # ── Loader Selection ─────────────────────────────────────────────────────
    # 'local'  → LocalFolderLoader (Task 1, fully implemented)
    # 'minio'  → MinioLoader       (Task 2 stub, not yet implemented)
    "loader"            : "local",
}

print("CONFIG loaded. Run ID:", CONFIG["run_id"])

CONFIG loaded. Run ID: 20260226_141843


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# A-1 | Package Check & Install
# Strategy (in order):
#   1. Skip if already importable.
#   2. Try `uv pip install` via the uv CLI (fastest, preferred).
#   3. Try `python -m uv pip install` (uv installed as a Python package).
#   4. Fall back to standard `pip install` (always available).
# ─────────────────────────────────────────────────────────────────────────────
import importlib
import shutil
import subprocess
import sys

REQUIRED_PACKAGES = {
    # import_name        : pip_package_name
    "tensorflow"         : "tensorflow[and-cuda]",
    "sklearn"            : "scikit-learn",
    "matplotlib"         : "matplotlib",
    "numpy"              : "numpy",
    "PIL"                : "Pillow",
}

def _is_package_available(import_name: str) -> bool:
    """Return True if the package can be imported."""
    return importlib.util.find_spec(import_name) is not None

def _run_cmd(cmd: list[str], label: str) -> bool:
    """
    Run a subprocess command. Return True on success, print stderr on failure.
    """
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        stderr_snippet = (result.stderr or result.stdout or "").strip()[:400]
        print(f"    [{label}] failed (exit {result.returncode}): {stderr_snippet}")
        return False
    return True

def _install_package(pip_package_name: str) -> None:
    """
    Install a package using the best available installer, with fallbacks.
    """
    install_args = ["pip", "install", pip_package_name, "-q"]

    # 1. uv CLI available globally (e.g. installed via `pip install uv` or system)
    uv_cli = shutil.which("uv")
    if uv_cli:
        print(f"  → Installing via uv CLI: {uv_cli} ...")
        if _run_cmd([uv_cli] + install_args, "uv-cli"):
            print(f"    ✓ Installed with uv CLI.")
            return

    # 2. uv available as a Python module inside this env
    print(f"  → Trying python -m uv pip install ...")
    if _run_cmd([sys.executable, "-m", "uv"] + install_args, "uv-module"):
        print(f"    ✓ Installed with uv module.")
        return

    # 3. Standard pip fallback (always present)
    print(f"  → Falling back to pip install {pip_package_name} ...")
    if _run_cmd([sys.executable, "-m", "pip", "install", pip_package_name, "-q"], "pip"):
        print(f"    ✓ Installed with pip.")
        return

    # All installers failed — warn but do not crash the notebook
    print(f"  ✗ WARNING: Could not install '{pip_package_name}'. "
          f"Install it manually before proceeding.")

print("Checking required packages ...")
for import_name, pip_name in REQUIRED_PACKAGES.items():
    if _is_package_available(import_name):
        print(f"  ✓ {import_name} already available")
    else:
        print(f"  ✗ {import_name} not found — installing {pip_name} ...")
        _install_package(pip_name)

print("\nAll required packages are present.")


Checking required packages ...
  ✓ tensorflow already available
  ✓ sklearn already available
  ✓ matplotlib already available
  ✓ numpy already available
  ✓ PIL already available

All required packages are present.


In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# A-2 | Core Imports
# ─────────────────────────────────────────────────────────────────────────────
import abc
import json
import os
import pathlib
import random
import shutil
import warnings
from collections import Counter, defaultdict
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")          # non-interactive backend; safe in Colab + scripts
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image

import sklearn.metrics as skmetrics
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore", category=UserWarning)
print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")

TensorFlow version : 2.20.0
Keras version      : 3.13.2


In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# A-3 | Deterministic Seeds, GPU Detection, Mixed Precision
# ─────────────────────────────────────────────────────────────────────────────

def setup_environment(cfg: dict) -> None:
    """Configure random seeds, GPU memory growth, and optional mixed precision."""
    seed = cfg["random_seed"]

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU(s) detected: {[g.name for g in gpus]}")
        except RuntimeError as exc:
            print(f"GPU setup warning: {exc}")
    else:
        print("No GPU detected — running on CPU (training will be slow).")

    if cfg["mixed_precision"] and gpus:
        keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision enabled: mixed_float16")
    else:
        keras.mixed_precision.set_global_policy("float32")
        print("Mixed precision disabled: float32")

setup_environment(CONFIG)

# ── Resolve REPO_ROOT ─────────────────────────────────────────────────────────
# Resolution priority (first that succeeds wins):
#
#   1. Explicit CONFIG['repo_root'] — not "." → used as-is (Colab / CI users).
#   2. __vsc_ipynb_file__  — VS Code injects this global with the absolute
#      path of the current .ipynb file.  The notebook lives at
#      <repo_root>/notebooks/<name>.ipynb so repo_root = parent.parent.
#   3. Walk up from pathlib.Path.cwd() looking for repo markers
#      (pyproject.toml, setup.py, .git, src/).  Falls back to cwd() itself.

def _find_repo_root_by_markers(start: pathlib.Path) -> pathlib.Path:
    """Walk up from `start` until a repo-root marker file/folder is found."""
    MARKERS = ("pyproject.toml", "setup.py", ".git")  # strict set — no ambiguous ones
    current = start.resolve()
    for _ in range(8):
        if any((current / m).exists() for m in MARKERS):
            return current
        parent = current.parent
        if parent == current:
            break
        current = parent
    return start.resolve()   # fallback

_cfg_root = CONFIG["repo_root"]

if _cfg_root != ".":
    # Priority 1 — Explicit path (Colab / Docker / CI)
    REPO_ROOT = pathlib.Path(_cfg_root).resolve()
    _root_source = "CONFIG['repo_root'] (explicit)"

else:
    # Priority 2 — VS Code injects __vsc_ipynb_file__ as the notebook's abs path
    try:
        _nb_path  = pathlib.Path(__vsc_ipynb_file__).resolve()  # noqa: F821
        # Notebook is at <repo_root>/notebooks/<name>.ipynb
        # so repo_root is exactly two levels up.
        _candidate = _nb_path.parent.parent
        # Sanity-check: at least one repo marker must exist at that location
        _MARKERS = ("pyproject.toml", "setup.py", ".git")
        if any((_candidate / m).exists() for m in _MARKERS):
            REPO_ROOT    = _candidate
            _root_source = f"__vsc_ipynb_file__ → {_nb_path.name}"
        else:
            # Marker not found two levels up — fall back to the marker walker
            REPO_ROOT    = _find_repo_root_by_markers(_nb_path.parent)
            _root_source = "__vsc_ipynb_file__ (marker walk)"
    except NameError:
        # Priority 3 — Not in VS Code: walk up from CWD
        REPO_ROOT    = _find_repo_root_by_markers(pathlib.Path.cwd())
        _root_source = "cwd() marker walk"

DISEASE_ROOT = REPO_ROOT / CONFIG["disease_folder"]
HEALTHY_ROOT = REPO_ROOT / CONFIG["healthy_folder"]
MODELS_DIR   = REPO_ROOT / CONFIG["models_dir"]
RUN_DIR      = REPO_ROOT / CONFIG["artifacts_dir"] / CONFIG["run_id"]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nRoot source  : {_root_source}")
print(f"Repo root    : {REPO_ROOT}")
print(f"Disease root : {DISEASE_ROOT}  ✓" if DISEASE_ROOT.exists() else
      f"Disease root : {DISEASE_ROOT}  ✗ NOT FOUND — check CONFIG paths")
print(f"Healthy root : {HEALTHY_ROOT}  ✓" if HEALTHY_ROOT.exists() else
      f"Healthy root : {HEALTHY_ROOT}  ✗ NOT FOUND — check CONFIG paths")
print(f"Run artifacts: {RUN_DIR}")

# Hard-stop if critical paths are missing — fail fast before training starts
if not DISEASE_ROOT.exists():
    raise FileNotFoundError(
        f"\nDISEASE_ROOT not found: {DISEASE_ROOT}\n"
        f"Fix: set CONFIG['repo_root'] to the absolute repo path, e.g.:\n"
        f"  CONFIG['repo_root'] = r'{REPO_ROOT.parent}'"  # suggest one level up
    )
if not HEALTHY_ROOT.exists():
    raise FileNotFoundError(
        f"\nHEALTHY_ROOT not found: {HEALTHY_ROOT}\n"
        f"Fix: set CONFIG['repo_root'] to the absolute repo path."
    )


No GPU detected — running on CPU (training will be slow).
Mixed precision disabled: float32

Root source  : __vsc_ipynb_file__ → tomato_disease_classifier_train.ipynb
Repo root    : E:\AgriTwin-GH
Disease root : E:\AgriTwin-GH\data\external\Tomato Diseases  ✓
Healthy root : E:\AgriTwin-GH\data\external\Tomato Healthy Leaves  ✓
Run artifacts: E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843


## Section B — Data Pipeline (Modular)

The pipeline uses a `DatasetLoader` interface.
- `LocalFolderLoader` — **fully implemented** (Task 1)
- `MinioLoader` — **stub only** (signatures + `NotImplementedError`, Task 2)

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# B-1 | DatasetLoader Abstract Interface
# ─────────────────────────────────────────────────────────────────────────────

class DatasetLoader(abc.ABC):
    """
    Abstract base class for dataset loaders.

    Any concrete loader MUST implement `load_file_label_pairs()` which
    returns a list of (absolute_image_path_str, label_str) tuples covering
    ALL splits (train + val + test combined).  The caller is responsible
    for splitting.

    This interface is intentionally minimal so swapping LocalFolderLoader
    for MinioLoader (Task 2) requires changing only one line.
    """

    @abc.abstractmethod
    def load_file_label_pairs(self) -> List[Tuple[str, str]]:
        """
        Returns:
            List of (image_path: str, label: str) tuples.
            image_path — absolute path to image file (JPEG/PNG).
            label      — canonical label string (from CONFIG folder_to_label).
        """
        ...

    @abc.abstractmethod
    def get_label_names(self) -> List[str]:
        """
        Returns:
            Sorted list of unique label strings produced by load_file_label_pairs.
        """
        ...

    @abc.abstractmethod
    def describe(self) -> str:
        """Human-readable description of the data source."""
        ...

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# B-2 | LocalFolderLoader — FULL IMPLEMENTATION
# ─────────────────────────────────────────────────────────────────────────────

class LocalFolderLoader(DatasetLoader):
    """
    Loads image file paths and labels from local extracted dataset folders.

    Handles TWO layout styles in a single scan:
      1. Sub-folder layout  — each subdirectory of `disease_root` is a class.
         Only folders in `include_folders` are used; others are skipped.
      2. Flat layout        — `healthy_root` itself contains images for one class
         (no sub-folders); the class label comes from `folder_to_label`.

    Args:
        disease_root    : Path to the Tomato Diseases folder.
        healthy_root    : Path to the Tomato Healthy Leaves folder.
        include_folders : List of sub-folder names to include from disease_root.
        folder_to_label : Dict mapping folder names → canonical label strings.
        image_exts      : Accepted image file extensions (lower-case).
    """

    VALID_EXTENSIONS: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

    def __init__(
        self,
        disease_root   : pathlib.Path,
        healthy_root   : pathlib.Path,
        include_folders: List[str],
        folder_to_label: Dict[str, str],
    ) -> None:
        self.disease_root    = pathlib.Path(disease_root)
        self.healthy_root    = pathlib.Path(healthy_root)
        self.include_folders = set(include_folders)
        self.folder_to_label = folder_to_label
        self._file_label_pairs: Optional[List[Tuple[str, str]]] = None

    # ── Internal helpers ──────────────────────────────────────────────────────

    @staticmethod
    def _collect_images_from_dir(directory: pathlib.Path) -> List[str]:
        """Recursively collect all image file paths under `directory`."""
        valid_exts = LocalFolderLoader.VALID_EXTENSIONS
        return [
            str(p.resolve())
            for p in directory.rglob("*")
            if p.is_file() and p.suffix.lower() in valid_exts
        ]

    def _scan_disease_root(self) -> List[Tuple[str, str]]:
        """Scan disease_root sub-folders → (path, label) pairs."""
        pairs: List[Tuple[str, str]] = []
        if not self.disease_root.exists():
            raise FileNotFoundError(f"Disease root not found: {self.disease_root}")

        for sub in sorted(self.disease_root.iterdir()):
            if not sub.is_dir():
                continue
            folder_name = sub.name
            if folder_name not in self.include_folders:
                print(f"  [SKIP] '{folder_name}' not in include_folders — skipping.")
                continue
            label = self.folder_to_label.get(folder_name)
            if label is None:
                print(f"  [WARN] No label mapping for '{folder_name}' — skipping.")
                continue
            images = self._collect_images_from_dir(sub)
            pairs.extend((img, label) for img in images)
            print(f"  [OK]   '{folder_name}' → '{label}'  ({len(images)} images)")
        return pairs

    def _scan_healthy_root(self) -> List[Tuple[str, str]]:
        """Scan healthy_root (flat layout) → (path, label) pairs."""
        if not self.healthy_root.exists():
            raise FileNotFoundError(f"Healthy root not found: {self.healthy_root}")
        label = self.folder_to_label.get(self.healthy_root.name)
        if label is None:
            raise KeyError(
                f"No label mapping for healthy folder '{self.healthy_root.name}'. "
                f"Add it to CONFIG['folder_to_label']."
            )
        images = self._collect_images_from_dir(self.healthy_root)
        print(f"  [OK]   '{self.healthy_root.name}' → '{label}'  ({len(images)} images)")
        return [(img, label) for img in images]

    # ── Public interface ──────────────────────────────────────────────────────

    def load_file_label_pairs(self) -> List[Tuple[str, str]]:
        """Scan local folders and return all (image_path, label) pairs."""
        if self._file_label_pairs is not None:
            return self._file_label_pairs   # cached — no re-scan

        print("Scanning disease sub-folders ...")
        disease_pairs  = self._scan_disease_root()
        print("Scanning healthy leaf folder ...")
        healthy_pairs  = self._scan_healthy_root()

        all_pairs = disease_pairs + healthy_pairs
        if not all_pairs:
            raise RuntimeError(
                "No images found.  Check that DISEASE_ROOT and HEALTHY_ROOT "
                "contain actual image files and that CONFIG paths are correct."
            )

        # Shuffle once with fixed seed for reproducibility
        rng = random.Random(42)
        rng.shuffle(all_pairs)
        self._file_label_pairs = all_pairs
        return self._file_label_pairs

    def get_label_names(self) -> List[str]:
        """Return sorted unique label strings."""
        pairs = self.load_file_label_pairs()
        return sorted(set(label for _, label in pairs))

    def describe(self) -> str:
        pairs  = self.load_file_label_pairs()
        counts = Counter(label for _, label in pairs)
        lines  = ["LocalFolderLoader summary:",
                  f"  Total images : {len(pairs)}"]
        for lbl in sorted(counts):
            lines.append(f"  {lbl:<35}: {counts[lbl]:>5}")
        return "\n".join(lines)

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# B-3 | MinioLoader — STUB (Task 2)
# Only method signatures and docstrings are provided.
# Replacing this stub with a real implementation in Task 2 requires
# NO changes outside this class.
# ─────────────────────────────────────────────────────────────────────────────

class MinioLoader(DatasetLoader):
    """
    (STUB — Task 2) Loads images from a MinIO object-storage backend.

    Downloads image bytes for each query result, writes them to a local
    temp cache, and returns (local_path, label) pairs so the rest of the
    pipeline is identical to LocalFolderLoader.

    Args:
        minio_endpoint    : MinIO server URL, e.g. 'localhost:9000'.
        bucket_name       : Name of the MinIO bucket containing images.
        label_prefix_map  : Dict mapping object-key prefix → label string.
                            e.g. {'diseases/early_blight/': 'tomato_early_blight'}
        access_key        : MinIO access key.
        secret_key        : MinIO secret key.
        local_cache_dir   : Local directory to cache downloaded images.
        secure            : Use HTTPS (default False for local dev).
    """

    def __init__(
        self,
        minio_endpoint  : str,
        bucket_name     : str,
        label_prefix_map: Dict[str, str],
        access_key      : str,
        secret_key      : str,
        local_cache_dir : str   = "/tmp/minio_image_cache",
        secure          : bool  = False,
    ) -> None:
        # TODO (Task 2): store args; initialise minio.Minio client
        raise NotImplementedError(
            "MinioLoader.__init__ is a stub.  "
            "Full implementation is scheduled for Task 2."
        )

    def load_file_label_pairs(self) -> List[Tuple[str, str]]:
        """
        (STUB) Query MinIO bucket, download images to local cache,
        and return (local_cached_path, label) pairs.

        Steps to implement in Task 2:
          1. For each prefix in label_prefix_map:
               client.list_objects(bucket, prefix=prefix, recursive=True)
          2. Download each object to local_cache_dir / prefix / filename
          3. Return (local_path, label) pairs
        """
        raise NotImplementedError("MinioLoader.load_file_label_pairs — Task 2 stub.")

    def get_label_names(self) -> List[str]:
        """
        (STUB) Return sorted unique labels derived from label_prefix_map values.
        """
        raise NotImplementedError("MinioLoader.get_label_names — Task 2 stub.")

    def describe(self) -> str:
        """
        (STUB) Return human-readable description of the MinIO data source.
        """
        raise NotImplementedError("MinioLoader.describe — Task 2 stub.")

    def _download_object(
        self,
        object_key   : str,
        local_path   : str,
    ) -> None:
        """
        (STUB) Download a single MinIO object to a local file.
        Task 2: client.fget_object(bucket, object_key, local_path)
        """
        raise NotImplementedError("MinioLoader._download_object — Task 2 stub.")

    def _build_local_cache(
        self,
        prefix       : str,
        label        : str,
    ) -> List[Tuple[str, str]]:
        """
        (STUB) List objects under `prefix`, download each, return (path, label) pairs.
        """
        raise NotImplementedError("MinioLoader._build_local_cache — Task 2 stub.")

print("DatasetLoader interface, LocalFolderLoader, and MinioLoader stub — defined.")

DatasetLoader interface, LocalFolderLoader, and MinioLoader stub — defined.


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# B-4 | Instantiate Loader + Load File-Label Pairs
# ─────────────────────────────────────────────────────────────────────────────

if CONFIG["loader"] == "local":
    loader: DatasetLoader = LocalFolderLoader(
        disease_root    = DISEASE_ROOT,
        healthy_root    = HEALTHY_ROOT,
        include_folders = CONFIG["include_disease_folders"],
        folder_to_label = CONFIG["folder_to_label"],
    )
elif CONFIG["loader"] == "minio":
    # Task 2: fill in MinIO credentials and bucket details
    loader = MinioLoader(
        minio_endpoint   = "localhost:9000",   # TODO: configure in Task 2
        bucket_name      = "agritwin-images",  # TODO: configure in Task 2
        label_prefix_map = {},                 # TODO: fill prefix → label map
        access_key       = "",                 # TODO: read from env / config
        secret_key       = "",
    )
else:
    raise ValueError(f"Unknown loader: {CONFIG['loader']!r}. Use 'local' or 'minio'.")

# Load all (path, label) pairs
all_file_label_pairs: List[Tuple[str, str]] = loader.load_file_label_pairs()
label_names: List[str] = loader.get_label_names()

# Integer label encoding
label_to_idx: Dict[str, int] = {lbl: i for i, lbl in enumerate(label_names)}
idx_to_label: Dict[int, str] = {i: lbl for lbl, i in label_to_idx.items()}

print("\n" + loader.describe())
print(f"\nLabel   → Index mapping:")
for lbl, idx in label_to_idx.items():
    print(f"  {idx}: {lbl}")

Scanning disease sub-folders ...
  [OK]   'Tomato_Early_Blight' → 'tomato_early_blight'  (1000 images)
  [OK]   'Tomato_Late_Blight' → 'tomato_late_blight'  (1909 images)
  [OK]   'Tomato_Leaf_Mold' → 'tomato_leaf_mold'  (3390 images)
  [OK]   'Tomato_Powdery_Mildew' → 'tomato_powdery_mildew'  (1256 images)
  [SKIP] 'Tomato_Septoria_Leaf_Spot' not in include_folders — skipping.
  [OK]   'Tomato_Spider_Mites' → 'tomato_spider_mites'  (2176 images)
Scanning healthy leaf folder ...
  [OK]   'Tomato Healthy Leaves' → 'tomato_leaf_healthy'  (1591 images)

LocalFolderLoader summary:
  Total images : 11322
  tomato_early_blight                :  1000
  tomato_late_blight                 :  1909
  tomato_leaf_healthy                :  1591
  tomato_leaf_mold                   :  3390
  tomato_powdery_mildew              :  1256
  tomato_spider_mites                :  2176

Label   → Index mapping:
  0: tomato_early_blight
  1: tomato_late_blight
  2: tomato_leaf_healthy
  3: tomato_leaf_mold
 

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# B-5 | Stratified Train / Val / Test Split
# Deterministic and reproducible; no dependency on pre-split folder structure.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

def stratified_three_way_split(
    file_label_pairs: List[Tuple[str, str]],
    val_fraction    : float,
    test_fraction   : float,
    seed            : int,
) -> Tuple[
    List[Tuple[str, str]],
    List[Tuple[str, str]],
    List[Tuple[str, str]],
]:
    """
    Split a flat list of (path, label) pairs into train / val / test subsets
    while preserving class proportions (stratified).

    Returns:
        train_pairs, val_pairs, test_pairs
    """
    paths  = [p for p, _ in file_label_pairs]
    labels = [l for _, l in file_label_pairs]

    # Step 1: carve off test set
    paths_tv, paths_test, labels_tv, labels_test = train_test_split(
        paths, labels,
        test_size    = test_fraction,
        stratify     = labels,
        random_state = seed,
    )

    # Step 2: carve val from remaining train+val
    val_fraction_adjusted = val_fraction / (1.0 - test_fraction)
    paths_train, paths_val, labels_train, labels_val = train_test_split(
        paths_tv, labels_tv,
        test_size    = val_fraction_adjusted,
        stratify     = labels_tv,
        random_state = seed,
    )

    train_pairs = list(zip(paths_train, labels_train))
    val_pairs   = list(zip(paths_val,   labels_val))
    test_pairs  = list(zip(paths_test,  labels_test))
    return train_pairs, val_pairs, test_pairs


train_pairs, val_pairs, test_pairs = stratified_three_way_split(
    file_label_pairs = all_file_label_pairs,
    val_fraction     = CONFIG["val_split"],
    test_fraction    = CONFIG["test_split"],
    seed             = CONFIG["random_seed"],
)

print(f"Split sizes — train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}")
print("Train class distribution:", Counter(l for _, l in train_pairs))

Split sizes — train: 8490, val: 1699, test: 1133
Train class distribution: Counter({'tomato_leaf_mold': 2542, 'tomato_spider_mites': 1631, 'tomato_late_blight': 1432, 'tomato_leaf_healthy': 1193, 'tomato_powdery_mildew': 942, 'tomato_early_blight': 750})


In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# B-6 | Class Weight Computation (handles class imbalance)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

def compute_class_weights(
    train_pairs: List[Tuple[str, str]],
    label_to_idx: Dict[str, int],
) -> Dict[int, float]:
    """Return class_weight dict keyed by integer class index."""
    train_labels_int = np.array([label_to_idx[l] for _, l in train_pairs])
    classes          = np.arange(len(label_to_idx))
    weights          = compute_class_weight(
        class_weight = "balanced",
        classes      = classes,
        y            = train_labels_int,
    )
    class_weight_dict = {int(cls): float(w) for cls, w in zip(classes, weights)}
    return class_weight_dict


CLASS_WEIGHTS = compute_class_weights(train_pairs, label_to_idx)

print("Class weights (higher → rarer class):")
for idx, weight in sorted(CLASS_WEIGHTS.items()):
    print(f"  {idx}: {idx_to_label[idx]:<35}  weight = {weight:.4f}")

Class weights (higher → rarer class):
  0: tomato_early_blight                  weight = 1.8867
  1: tomato_late_blight                   weight = 0.9881
  2: tomato_leaf_healthy                  weight = 1.1861
  3: tomato_leaf_mold                     weight = 0.5566
  4: tomato_powdery_mildew                weight = 1.5021
  5: tomato_spider_mites                  weight = 0.8676


In [34]:

# ─────────────────────────────────────────────────────────────────────────────
# B-7 | tf.data Pipeline — Preprocessing & Augmentation
# ─────────────────────────────────────────────────────────────────────────────

IMG_H, IMG_W = CONFIG["image_size"]
NUM_CLASSES  = CONFIG["num_classes"]

# ── Backbone normalization function (maps [0,255] → backbone-specific range) ─
BACKBONE_PREPROCESS_MAP = {
    "EfficientNetB0"   : keras.applications.efficientnet.preprocess_input,
    "EfficientNetB3"   : keras.applications.efficientnet.preprocess_input,
    "ResNet50"         : keras.applications.resnet.preprocess_input,
    "MobileNetV3Large" : keras.applications.mobilenet_v3.preprocess_input,
    "DenseNet121"      : keras.applications.densenet.preprocess_input,
}

_backbone_preprocess_fn = BACKBONE_PREPROCESS_MAP.get(CONFIG["backbone_name"])
if _backbone_preprocess_fn is None:
    raise KeyError(
        f"Backbone '{CONFIG['backbone_name']}' not in BACKBONE_PREPROCESS_MAP. "
        f"Add its preprocess_input function."
    )

# ── Keras augmentation layers — instantiated ONCE at cell scope so they are  ─
#    never created inside a tf.function trace (which would raise ValueError    ─
#    because tf.Variable creation inside tf.function is forbidden).            ─
_rotation_layer = keras.layers.RandomRotation(
    factor    = CONFIG["aug_rotation_factor"],
    fill_mode = "reflect",
    seed      = None,
)
_zoom_layer = keras.layers.RandomZoom(
    height_factor = (-CONFIG["aug_zoom_factor"], CONFIG["aug_zoom_factor"]),
    fill_mode     = "reflect",
    seed          = None,
)

# ── Cutout / Random Erasing (custom layer — framework-native, no extra deps) ─

def apply_cutout(
    image       : tf.Tensor,
    cutout_frac : float,
) -> tf.Tensor:
    """
    Randomly erase a square patch of `cutout_frac` × min(H,W) pixels
    and fill with the image mean colour.
    Applied with 50 % probability per image.
    """
    # 50 % chance to apply
    if tf.random.uniform(()) > 0.5:
        return image

    h, w = tf.shape(image)[0], tf.shape(image)[1]
    patch_size = tf.cast(tf.cast(tf.minimum(h, w), tf.float32) * cutout_frac, tf.int32)

    # Random top-left corner
    top  = tf.random.uniform((), 0, h - patch_size, dtype=tf.int32)
    left = tf.random.uniform((), 0, w - patch_size, dtype=tf.int32)

    mean_val = tf.reduce_mean(image)

    # Build mask: 1 where cutout, 0 elsewhere
    row_mask = tf.logical_and(
        tf.range(h) >= top,
        tf.range(h) < (top + patch_size)
    )
    col_mask = tf.logical_and(
        tf.range(w) >= left,
        tf.range(w) < (left + patch_size)
    )
    # Outer product → 2-D patch mask, then broadcast to [H, W, C]
    patch_mask = tf.cast(
        tf.logical_and(
            tf.expand_dims(row_mask, 1),
            tf.expand_dims(col_mask, 0)
        ),
        image.dtype
    )
    patch_mask = tf.expand_dims(patch_mask, -1)  # [H, W, 1]
    image = image * (1.0 - patch_mask) + mean_val * patch_mask
    return image


# ── Augmentation pipeline (training only) ────────────────────────────────────

def augment_image(image: tf.Tensor, cfg: dict) -> tf.Tensor:
    """
    Apply a sequence of randomized augmentations to a UINT8 image tensor.
    Uses module-level layer objects (_rotation_layer, _zoom_layer) so that
    no tf.Variable is created inside the tf.function trace.
    """
    image = tf.cast(image, tf.float32)

    # 1. Random horizontal + vertical flip
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    # 2. Random rotation — uses the pre-built layer (no new Variable created)
    image = _rotation_layer(tf.expand_dims(image, 0), training=True)[0]

    # 3. Random zoom — uses the pre-built layer (no new Variable created)
    image = _zoom_layer(tf.expand_dims(image, 0), training=True)[0]

    # 4. Random brightness
    image = tf.image.random_brightness(image, max_delta=cfg["aug_brightness_delta"] * 255.0)

    # 5. Random contrast
    lower_c = 1.0 - cfg["aug_contrast_factor"]
    upper_c = 1.0 + cfg["aug_contrast_factor"]
    image = tf.image.random_contrast(image, lower=lower_c, upper=upper_c)

    # 6. Random hue  (subtle — keep leaf colour plausible)
    image = tf.image.random_hue(image / 255.0, max_delta=cfg["aug_hue_delta"]) * 255.0

    # 7. Random saturation
    image = tf.image.random_saturation(
        image / 255.0,
        lower = cfg["aug_saturation_lower"],
        upper = cfg["aug_saturation_upper"],
    ) * 255.0

    # 8. Random crop (retains aug_crop_fraction of each side)
    crop_frac   = cfg["aug_crop_fraction"]
    crop_h      = tf.cast(tf.shape(image)[0], tf.float32)
    crop_w      = tf.cast(tf.shape(image)[1], tf.float32)
    crop_size_h = tf.cast(crop_h * crop_frac, tf.int32)
    crop_size_w = tf.cast(crop_w * crop_frac, tf.int32)
    image = tf.image.random_crop(image, size=[crop_size_h, crop_size_w, 3])
    image = tf.image.resize(image, [IMG_H, IMG_W])

    # 9. Random cutout / erasing
    image = apply_cutout(image, cfg["aug_cutout_fraction"])

    # Clip to valid pixel range before backbone normalisation
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image


# ── Core image loading function ───────────────────────────────────────────────

def load_and_resize_image(path: str) -> tf.Tensor:
    """Read image from disk, decode, and resize to (IMG_H, IMG_W, 3)."""
    raw    = tf.io.read_file(path)
    image  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image  = tf.image.resize(image, [IMG_H, IMG_W])
    image  = tf.cast(image, tf.float32)
    return image


# ── tf.data map functions ─────────────────────────────────────────────────────

def parse_and_augment(path: tf.Tensor, label_idx: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Used for TRAINING set: load → augment → normalise → one-hot."""
    image      = load_and_resize_image(path)
    image      = augment_image(image, CONFIG)
    image      = _backbone_preprocess_fn(image)   # backbone-specific normalisation
    label_ohe  = tf.one_hot(label_idx, NUM_CLASSES)
    return image, label_ohe

def parse_only(path: tf.Tensor, label_idx: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Used for VAL / TEST sets: load → normalise → one-hot (no augmentation)."""
    image      = load_and_resize_image(path)
    image      = _backbone_preprocess_fn(image)
    label_ohe  = tf.one_hot(label_idx, NUM_CLASSES)
    return image, label_ohe


# ── Build tf.data Dataset objects ────────────────────────────────────────────

def pairs_to_tf_dataset(
    pairs       : List[Tuple[str, str]],
    label_to_idx: Dict[str, int],
    map_fn,
    batch_size  : int,
    shuffle     : bool = False,
    seed        : int  = 42,
) -> tf.data.Dataset:
    """Convert a list of (path, label) pairs into a batched tf.data.Dataset."""
    paths      = [p for p, _ in pairs]
    label_idxs = [label_to_idx[l] for _, l in pairs]

    ds = tf.data.Dataset.from_tensor_slices((paths, label_idxs))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(pairs), seed=seed, reshuffle_each_iteration=True)

    AUTOTUNE = tf.data.AUTOTUNE
    ds = ds.map(map_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds


BATCH_SIZE = CONFIG["batch_size"]
SEED       = CONFIG["random_seed"]

train_ds = pairs_to_tf_dataset(train_pairs, label_to_idx, parse_and_augment, BATCH_SIZE, shuffle=True,  seed=SEED)
val_ds   = pairs_to_tf_dataset(val_pairs,   label_to_idx, parse_only,        BATCH_SIZE, shuffle=False, seed=SEED)
test_ds  = pairs_to_tf_dataset(test_pairs,  label_to_idx, parse_only,        BATCH_SIZE, shuffle=False, seed=SEED)

print(f"train_ds  : {len(train_pairs)} samples, {len(train_ds)} batches")
print(f"val_ds    : {len(val_pairs)}   samples, {len(val_ds)} batches")
print(f"test_ds   : {len(test_pairs)}  samples, {len(test_ds)} batches")


train_ds  : 8490 samples, 266 batches
val_ds    : 1699   samples, 54 batches
test_ds   : 1133  samples, 36 batches


## Section C — Model Architecture

In [35]:
# ─────────────────────────────────────────────────────────────────────────────
# C-1 | Backbone Factory
# ─────────────────────────────────────────────────────────────────────────────

def build_backbone(
    name       : str,
    input_shape: Tuple[int, int, int],
) -> keras.Model:
    """
    Instantiate a pretrained backbone (ImageNet weights) with the top
    classification head removed.  The backbone is returned with
    ALL layers frozen — Phase 1 (warm-up) training only updates the
    custom head added on top.

    Args:
        name        : One of the supported backbone strings in BACKBONE_PREPROCESS_MAP.
        input_shape : (H, W, C) — must match CONFIG['image_size'] + channels.

    Returns:
        Keras functional model (feature extractor only, include_top=False).
    """
    common_kwargs = dict(
        include_top = False,
        weights     = "imagenet",
        input_shape = input_shape,
    )

    backbone_constructors = {
        "EfficientNetB0"   : keras.applications.EfficientNetB0,
        "EfficientNetB3"   : keras.applications.EfficientNetB3,
        "ResNet50"         : keras.applications.ResNet50,
        "MobileNetV3Large" : keras.applications.MobileNetV3Large,
        "DenseNet121"      : keras.applications.DenseNet121,
    }

    if name not in backbone_constructors:
        raise ValueError(f"Unsupported backbone: '{name}'. Choose from {list(backbone_constructors)}.")

    backbone = backbone_constructors[name](**common_kwargs)
    backbone.trainable = False   # freeze all layers for warm-up phase
    print(f"Backbone '{name}' loaded — {len(backbone.layers)} layers, all frozen.")
    return backbone


# ── Build full model (backbone + custom classification head) ──────────────────

def build_classifier(
    backbone_name: str,
    image_size   : Tuple[int, int],
    num_classes  : int,
    dropout_1    : float,
    dense_units  : int,
    dropout_2    : float,
) -> keras.Model:
    """
    Assemble the full classification model:
        Input → Backbone (frozen) → GlobalAveragePooling2D
             → Dropout(dropout_1) → Dense(dense_units, relu) → BatchNorm
             → Dropout(dropout_2) → Dense(num_classes, softmax)

    Returns:
        Compiled Keras Model (not yet compiled — compilation happens later
        so we can set different LRs per phase).
    """
    H, W = image_size
    inputs   = keras.Input(shape=(H, W, 3), name="image_input")
    backbone = build_backbone(backbone_name, input_shape=(H, W, 3))

    x = backbone(inputs, training=False)   # training=False → BN in inference mode during warm-up

    # Custom classification head
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(dropout_1, name="dropout_1")(x)
    x = keras.layers.Dense(dense_units, activation="relu", name="head_dense")(x)
    x = keras.layers.BatchNormalization(name="head_bn")(x)
    x = keras.layers.Dropout(dropout_2, name="dropout_2")(x)

    # Output layer — dtype=float32 explicit for mixed-precision stability
    outputs = keras.layers.Dense(
        num_classes,
        activation = "softmax",
        dtype      = "float32",
        name       = "predictions",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name=f"{backbone_name}_classifier")
    return model, backbone


model, backbone = build_classifier(
    backbone_name = CONFIG["backbone_name"],
    image_size    = CONFIG["image_size"],
    num_classes   = CONFIG["num_classes"],
    dropout_1     = CONFIG["head_dropout_1"],
    dense_units   = CONFIG["head_units"],
    dropout_2     = CONFIG["head_dropout_2"],
)

model.summary(line_length=90)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step
Backbone 'EfficientNetB0' loaded — 238 layers, all frozen.


Model: "EfficientNetB0_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)              │ (None, 224, 224, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ efficientnetb0 (Functional)           │ (None, 7, 7, 1280)           │       4,049,571 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ gap (GlobalAveragePooling2D)          │ (None, 1280)                 │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                   │ (None, 1280)                 │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ head_dense (Dense)                    │ (None, 256)                  │         327,936 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ head_bn (BatchNormalization)          │ (None, 256)                  │           1,024 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                   │ (None, 256)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ predictions (Dense)                   │ (None, 6)                    │           1,542 │
└───────────────────────────────────────┴──────────────────────────────┴─────────────────┘

 Total params: 4,380,073 (16.71 MB)

 Trainable params: 329,990 (1.26 MB)

 Non-trainable params: 4,050,083 (15.45 MB)

In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# C-2 | Loss Function Factory
# ─────────────────────────────────────────────────────────────────────────────

def build_loss_fn(loss_type: str, cfg: dict):
    """
    Return a Keras loss function object based on CONFIG['loss_type'].

    'ce'    → CategoricalCrossentropy with label smoothing.
    'focal' → Sigmoid Focal Cross-Entropy (custom implementation).
              Useful when class imbalance is extreme.
    """
    if loss_type == "ce":
        loss_fn = keras.losses.CategoricalCrossentropy(
            label_smoothing = cfg["label_smoothing"],
            from_logits     = False,
        )
        print(f"Loss: CategoricalCrossentropy (label_smoothing={cfg['label_smoothing']})")
        return loss_fn

    elif loss_type == "focal":
        alpha = cfg["focal_alpha"]
        gamma = cfg["focal_gamma"]

        def focal_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
            """
            Multiclass Focal Loss (One-vs-Rest decomposition).
            y_true: one-hot  [B, C]
            y_pred: softmax  [B, C]
            """
            y_pred = tf.cast(tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7), tf.float32)
            y_true = tf.cast(y_true, tf.float32)
            ce     = -y_true * tf.math.log(y_pred)
            p_t    = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
            focal_weight = alpha * tf.pow(1.0 - p_t, gamma)
            loss   = focal_weight * tf.reduce_sum(ce, axis=-1)
            return tf.reduce_mean(loss)

        print(f"Loss: Focal (alpha={alpha}, gamma={gamma})")
        return focal_loss

    else:
        raise ValueError(f"Unknown loss_type: '{loss_type}'. Use 'ce' or 'focal'.")


loss_fn = build_loss_fn(CONFIG["loss_type"], CONFIG)

Loss: CategoricalCrossentropy (label_smoothing=0.1)


In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# C-3 | Callbacks
# ─────────────────────────────────────────────────────────────────────────────

HISTORY_CSV_PATH      = str(RUN_DIR / "training_history.csv")
CHECKPOINT_PATH       = str(MODELS_DIR / f"{CONFIG['run_id']}_best.keras")

def build_callbacks(
    phase       : str,   # 'warmup' | 'finetune'
    checkpoint_path: str,
    history_csv_path: str,
) -> List[keras.callbacks.Callback]:
    """
    Build a standard callback set for one training phase.
    CSVLogger is set to append=True so both phases share one file.
    """
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor              = "val_loss",
            patience             = 6,
            restore_best_weights = True,
            verbose              = 1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor  = "val_loss",
            factor   = 0.4,
            patience = 3,
            min_lr   = 1e-7,
            verbose  = 1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath           = checkpoint_path,
            monitor            = "val_accuracy",
            save_best_only     = True,
            save_weights_only  = False,
            verbose            = 1,
        ),
        keras.callbacks.CSVLogger(
            filename = history_csv_path,
            append   = True,       # append so both phases are in one file
        ),
    ]
    print(f"Callbacks built for phase '{phase}'.")
    return callbacks

print(f"Checkpoint path  : {CHECKPOINT_PATH}")
print(f"History CSV path : {HISTORY_CSV_PATH}")

Checkpoint path  : E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras
History CSV path : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\training_history.csv


## Section D — Training (Two-Phase: Warm-Up → Fine-Tune)

In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# D-1 | Phase 1 — Warm-Up Training (backbone FROZEN)
# Only the custom head is trained.  Fast convergence.
# ─────────────────────────────────────────────────────────────────────────────

print("═" * 60)
print("PHASE 1 — WARM-UP (backbone frozen)")
print("═" * 60)

# Ensure backbone is frozen
backbone.trainable = False
print(f"Trainable params (warmup): {model.count_params():,}")

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=CONFIG["lr_warmup"]),
    loss      = loss_fn,
    metrics   = ["accuracy"],
)

warmup_callbacks = build_callbacks(
    phase            = "warmup",
    checkpoint_path  = CHECKPOINT_PATH,
    history_csv_path = HISTORY_CSV_PATH,
)

warmup_history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = CONFIG["epochs_warmup"],
    class_weight    = CLASS_WEIGHTS,
    callbacks       = warmup_callbacks,
    verbose         = 1,
)

print(f"\nWarm-up complete. "
      f"Best val_accuracy = {max(warmup_history.history['val_accuracy']):.4f}")

════════════════════════════════════════════════════════════
PHASE 1 — WARM-UP (backbone frozen)
════════════════════════════════════════════════════════════
Trainable params (warmup): 4,380,073
Callbacks built for phase 'warmup'.
Epoch 1/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6528 - loss: 1.2632
Epoch 1: val_accuracy improved from None to 0.84991, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras

Epoch 1: finished saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras
266/266 ━━━━━━━━━━━━━━━━━━━━ 891s 3s/step - accuracy: 0.7486 - loss: 1.0290 - val_accuracy: 0.8499 - val_loss: 0.7615 - learning_rate: 0.0010
Epoch 2/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8388 - loss: 0.8025
Epoch 2: val_accuracy improved from 0.84991 to 0.85756, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras

Epoch 2: finished saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_1418

In [39]:
# ─────────────────────────────────────────────────────────────────────────────
# D-2 | Phase 2 — Fine-Tuning (top N backbone layers UNFROZEN)
# ─────────────────────────────────────────────────────────────────────────────

print("═" * 60)
print("PHASE 2 — FINE-TUNING (top backbone layers unfrozen)")
print("═" * 60)

UNFREEZE_TOP_N = CONFIG["unfreeze_top_layers"]

# Unfreeze the top N layers of the backbone
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_TOP_N]:
    layer.trainable = False

# Count BatchNorm layers — keep them in inference mode to avoid instability
for layer in backbone.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

n_trainable = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after unfreeze : {n_trainable}")
print(f"Trainable params (finetune)     : {model.count_params():,}")

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=CONFIG["lr_finetune"]),
    loss      = loss_fn,
    metrics   = ["accuracy"],
)

finetune_callbacks = build_callbacks(
    phase            = "finetune",
    checkpoint_path  = CHECKPOINT_PATH,
    history_csv_path = HISTORY_CSV_PATH,
)

finetune_history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = CONFIG["epochs_finetune"],
    class_weight    = CLASS_WEIGHTS,
    callbacks       = finetune_callbacks,
    verbose         = 1,
)

print(f"\nFine-tuning complete. "
      f"Best val_accuracy = {max(finetune_history.history['val_accuracy']):.4f}")

════════════════════════════════════════════════════════════
PHASE 2 — FINE-TUNING (top backbone layers unfrozen)
════════════════════════════════════════════════════════════
Trainable layers after unfreeze : 8
Trainable params (finetune)     : 4,380,073
Callbacks built for phase 'finetune'.
Epoch 1/25
266/266 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9081 - loss: 0.6609
Epoch 1: val_accuracy improved from None to 0.92643, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras

Epoch 1: finished saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras
266/266 ━━━━━━━━━━━━━━━━━━━━ 987s 4s/step - accuracy: 0.9160 - loss: 0.6446 - val_accuracy: 0.9264 - val_loss: 0.6218 - learning_rate: 5.0000e-05
Epoch 2/25
266/266 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9418 - loss: 0.6089
Epoch 2: val_accuracy improved from 0.92643 to 0.93584, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras

Epoch 2: finished s

In [40]:
# ─────────────────────────────────────────────────────────────────────────────
# D-3 | Training History Plot (visual sanity check)
# ─────────────────────────────────────────────────────────────────────────────

# Merge warmup + finetune histories
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

merged_hist = merge_histories(warmup_history, finetune_history)
warmup_end  = len(warmup_history.history["loss"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_total = range(1, len(merged_hist["loss"]) + 1)

# Loss plot
axes[0].plot(epochs_total, merged_hist["loss"],     label="Train Loss")
axes[0].plot(epochs_total, merged_hist["val_loss"], label="Val Loss")
axes[0].axvline(warmup_end + 0.5, color="grey", linestyle="--", label="Warm-up / Fine-tune")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

# Accuracy plot
axes[1].plot(epochs_total, merged_hist["accuracy"],     label="Train Acc")
axes[1].plot(epochs_total, merged_hist["val_accuracy"], label="Val Acc")
axes[1].axvline(warmup_end + 0.5, color="grey", linestyle="--", label="Warm-up / Fine-tune")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.suptitle(f"Training History — Run: {CONFIG['run_id']}")
plt.tight_layout()
HISTORY_PLOT_PATH = str(RUN_DIR / "training_history_plot.png")
plt.savefig(HISTORY_PLOT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"History plot saved → {HISTORY_PLOT_PATH}")

History plot saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\training_history_plot.png


## Section E — Evaluation Metrics

In [41]:
# ─────────────────────────────────────────────────────────────────────────────
# E-1 | Load Best Checkpoint + Run Inference on Test Set
# ─────────────────────────────────────────────────────────────────────────────

print(f"Loading best checkpoint from: {CHECKPOINT_PATH}")
best_model = keras.models.load_model(CHECKPOINT_PATH)

# Collect ground-truth labels and model predictions
y_true_int  = []   # integer labels
y_pred_prob = []   # softmax probabilities [N, num_classes]

for batch_images, batch_labels_ohe in test_ds:
    probs = best_model.predict_on_batch(batch_images)        # [B, C]
    y_pred_prob.append(probs)
    y_true_int.extend(np.argmax(batch_labels_ohe.numpy(), axis=1))

y_pred_prob = np.vstack(y_pred_prob)              # [N, C]
y_true_int  = np.array(y_true_int)                # [N]
y_pred_int  = np.argmax(y_pred_prob, axis=1)      # [N]

print(f"Test samples evaluated: {len(y_true_int)}")
print(f"Test accuracy (raw): {np.mean(y_true_int == y_pred_int):.4f}")

Loading best checkpoint from: E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras
Test samples evaluated: 1133
Test accuracy (raw): 0.9823


In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# E-2 | Compute All Metrics
# ─────────────────────────────────────────────────────────────────────────────

def compute_all_metrics(
    y_true      : np.ndarray,
    y_pred      : np.ndarray,
    y_pred_prob : np.ndarray,
    label_names : List[str],
) -> dict:
    """
    Compute a comprehensive set of evaluation metrics.

    Returns:
        metrics dict with all scalar and per-class values.
    """
    num_classes = len(label_names)

    # Basic
    overall_acc  = float(skmetrics.accuracy_score(y_true, y_pred))
    prec_macro   = float(skmetrics.precision_score(y_true, y_pred, average="macro",    zero_division=0))
    rec_macro    = float(skmetrics.recall_score   (y_true, y_pred, average="macro",    zero_division=0))
    f1_macro     = float(skmetrics.f1_score       (y_true, y_pred, average="macro",    zero_division=0))

    prec_weighted = float(skmetrics.precision_score(y_true, y_pred, average="weighted", zero_division=0))
    rec_weighted  = float(skmetrics.recall_score   (y_true, y_pred, average="weighted", zero_division=0))
    f1_weighted   = float(skmetrics.f1_score       (y_true, y_pred, average="weighted", zero_division=0))

    # Per-class accuracy
    cm          = skmetrics.confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    per_class_acc = {}
    for i, lbl in enumerate(label_names):
        tp        = cm[i, i]
        total     = cm[i, :].sum()
        per_class_acc[lbl] = float(tp / total) if total > 0 else 0.0

    # ROC-AUC One-vs-Rest
    try:
        y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
        roc_auc_macro = float(
            skmetrics.roc_auc_score(y_true_bin, y_pred_prob, average="macro", multi_class="ovr")
        )
        roc_auc_weighted = float(
            skmetrics.roc_auc_score(y_true_bin, y_pred_prob, average="weighted", multi_class="ovr")
        )
    except Exception as exc:
        print(f"  [WARN] ROC-AUC computation failed: {exc}")
        roc_auc_macro    = None
        roc_auc_weighted = None

    # Classification report (text)
    clf_report_text = skmetrics.classification_report(
        y_true,
        y_pred,
        target_names = label_names,
        zero_division = 0,
    )

    metrics = {
        "overall_accuracy"       : overall_acc,
        "precision_macro"        : prec_macro,
        "recall_macro"           : rec_macro,
        "f1_macro"               : f1_macro,
        "precision_weighted"     : prec_weighted,
        "recall_weighted"        : rec_weighted,
        "f1_weighted"            : f1_weighted,
        "roc_auc_macro_ovr"      : roc_auc_macro,
        "roc_auc_weighted_ovr"   : roc_auc_weighted,
        "per_class_accuracy"     : per_class_acc,
        "classification_report"  : clf_report_text,
        "num_test_samples"        : int(len(y_true)),
        "run_id"                 : CONFIG["run_id"],
        "backbone"               : CONFIG["backbone_name"],
    }
    return metrics


eval_metrics = compute_all_metrics(
    y_true      = y_true_int,
    y_pred      = y_pred_int,
    y_pred_prob = y_pred_prob,
    label_names = label_names,
)

print("\n── Test Set Metrics ──────────────────────────────────────────")
for key, value in eval_metrics.items():
    if key in ("classification_report", "per_class_accuracy"):
        continue
    print(f"  {key:<28}: {value}")
print("\nPer-class accuracy:")
for lbl, acc in eval_metrics["per_class_accuracy"].items():
    print(f"  {lbl:<35}: {acc:.4f}")
print("\nClassification Report:")
print(eval_metrics["classification_report"])


── Test Set Metrics ──────────────────────────────────────────
  overall_accuracy            : 0.9823477493380406
  precision_macro             : 0.9804156652993862
  recall_macro                : 0.9867255735072574
  f1_macro                    : 0.9833938861394134
  precision_weighted          : 0.9827135436774387
  recall_weighted             : 0.9823477493380406
  f1_weighted                 : 0.9823138843806887
  roc_auc_macro_ovr           : 0.999914488933698
  roc_auc_weighted_ovr        : 0.9999138706571596
  num_test_samples            : 1133
  run_id                      : 20260226_141843
  backbone                    : EfficientNetB0

Per-class accuracy:
  tomato_early_blight                : 0.9800
  tomato_late_blight                 : 1.0000
  tomato_leaf_healthy                : 1.0000
  tomato_leaf_mold                   : 0.9587
  tomato_powdery_mildew              : 1.0000
  tomato_spider_mites                : 0.9817

Classification Report:
                       pr

In [43]:
# ─────────────────────────────────────────────────────────────────────────────
# E-3 | Confusion Matrix Plot
# ─────────────────────────────────────────────────────────────────────────────

def plot_confusion_matrix(
    y_true      : np.ndarray,
    y_pred      : np.ndarray,
    label_names : List[str],
    save_path   : str,
    title       : str = "Confusion Matrix",
) -> None:
    """Plot a normalised and raw confusion matrix side-by-side and save as PNG."""
    num_classes = len(label_names)
    cm_raw  = skmetrics.confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    short_names = [lbl.replace("tomato_", "") for lbl in label_names]

    for ax, data, fmt, subtitle in [
        (axes[0], cm_norm, ".2f", "Normalised (row %)"),
        (axes[1], cm_raw,  "d",   "Raw Counts"),
    ]:
        im = ax.imshow(data, interpolation="nearest", cmap="Blues",
                       vmin=0, vmax=(1.0 if fmt == ".2f" else cm_raw.max()))
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set(xticks=range(num_classes), yticks=range(num_classes),
               xticklabels=short_names, yticklabels=short_names,
               xlabel="Predicted", ylabel="True",
               title=subtitle)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
        thresh = data.max() / 2.0
        for i in range(num_classes):
            for j in range(num_classes):
                ax.text(j, i, format(data[i, j], fmt),
                        ha="center", va="center",
                        color="white" if data[i, j] > thresh else "black",
                        fontsize=8)

    plt.suptitle(f"{title} — Run: {CONFIG['run_id']}", fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Confusion matrix saved → {save_path}")


CM_SAVE_PATH = str(RUN_DIR / "confusion_matrix.png")
plot_confusion_matrix(
    y_true      = y_true_int,
    y_pred      = y_pred_int,
    label_names = label_names,
    save_path   = CM_SAVE_PATH,
)

Confusion matrix saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\confusion_matrix.png


In [44]:
# ─────────────────────────────────────────────────────────────────────────────
# E-4 | ROC Curves (One-vs-Rest, per class)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(
    y_true      : np.ndarray,
    y_pred_prob : np.ndarray,
    label_names : List[str],
    save_path   : str,
) -> None:
    """Plot One-vs-Rest ROC curve for each class and save as PNG."""
    num_classes   = len(label_names)
    y_true_bin    = label_binarize(y_true, classes=list(range(num_classes)))

    fig, ax = plt.subplots(figsize=(9, 7))
    colours = plt.cm.tab10(np.linspace(0, 1, num_classes))

    for i, (lbl, colour) in enumerate(zip(label_names, colours)):
        fprs, tprs, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
        roc_auc       = auc(fprs, tprs)
        ax.plot(fprs, tprs, color=colour,
                label=f"{lbl.replace('tomato_', '')} (AUC={roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set(xlim=[0, 1], ylim=[0, 1.02],
           xlabel="False Positive Rate",
           ylabel="True Positive Rate",
           title=f"ROC Curves (OvR) — Run: {CONFIG['run_id']}")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"ROC curve plot saved → {save_path}")


ROC_SAVE_PATH = str(RUN_DIR / "roc_curves.png")
try:
    plot_roc_curves(
        y_true      = y_true_int,
        y_pred_prob = y_pred_prob,
        label_names = label_names,
        save_path   = ROC_SAVE_PATH,
    )
except Exception as exc:
    print(f"[WARN] ROC plot skipped: {exc}")

ROC curve plot saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\roc_curves.png


In [45]:
# ─────────────────────────────────────────────────────────────────────────────
# E-5 | Misclassified Examples Grid (≥20 images)
# ─────────────────────────────────────────────────────────────────────────────

def build_misclassified_grid(
    test_pairs  : List[Tuple[str, str]],
    y_true      : np.ndarray,
    y_pred      : np.ndarray,
    y_pred_prob : np.ndarray,
    idx_to_label: Dict[int, str],
    save_path   : str,
    n_show      : int = 20,
    thumb_size  : Tuple[int, int] = (128, 128),
) -> None:
    """
    Collect misclassified samples, sort by prediction confidence
    (highest confidence wrong → most instructive), and render a grid PNG.

    The grid caption for each image shows:
        True: <label>\nPredicted: <label> (conf%)
    """
    # Find misclassified indices (aligned with test_pairs order)
    misclassified_indices = np.where(y_true != y_pred)[0]

    if len(misclassified_indices) == 0:
        print("[INFO] No misclassified samples — perfect test accuracy!")
        return

    # Sort by descending confidence of the wrong prediction
    wrong_conf   = y_pred_prob[misclassified_indices, y_pred[misclassified_indices]]
    sorted_order = np.argsort(-wrong_conf)       # descending
    selected_idx = misclassified_indices[sorted_order[:n_show]]

    n_cols = 5
    n_rows = max((len(selected_idx) + n_cols - 1) // n_cols, 1)
    fig    = plt.figure(figsize=(n_cols * 3, n_rows * 3.6))
    fig.suptitle(
        f"Top-{len(selected_idx)} Misclassified Samples (by wrong pred confidence)\n"
        f"Run: {CONFIG['run_id']}",
        fontsize=12,
    )

    for plot_pos, sample_idx in enumerate(selected_idx, start=1):
        img_path  = test_pairs[sample_idx][0]
        true_lbl  = idx_to_label[int(y_true[sample_idx])].replace("tomato_", "")
        pred_lbl  = idx_to_label[int(y_pred[sample_idx])].replace("tomato_", "")
        conf_pct  = y_pred_prob[sample_idx, y_pred[sample_idx]] * 100

        try:
            img = Image.open(img_path).convert("RGB").resize(thumb_size)
        except Exception:
            img = Image.new("RGB", thumb_size, color=(200, 200, 200))

        ax = fig.add_subplot(n_rows, n_cols, plot_pos)
        ax.imshow(img)
        ax.set_title(
            f"True: {true_lbl}\nPred: {pred_lbl} ({conf_pct:.1f}%)",
            fontsize=7.5,
            color="red",
        )
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Misclassified grid saved → {save_path}")
    print(f"Total misclassified: {len(misclassified_indices)} / {len(y_true)}")


MISCLASSIFIED_SAVE_PATH = str(RUN_DIR / "misclassified_grid.png")
build_misclassified_grid(
    test_pairs   = test_pairs,
    y_true       = y_true_int,
    y_pred       = y_pred_int,
    y_pred_prob  = y_pred_prob,
    idx_to_label = idx_to_label,
    save_path    = MISCLASSIFIED_SAVE_PATH,
    n_show       = 25,    # show up to 25 (≥20 as required)
)

Misclassified grid saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\misclassified_grid.png
Total misclassified: 20 / 1133


## Section F — Save All Artifacts

In [46]:
# ─────────────────────────────────────────────────────────────────────────────
# F-1 | Save Model (.keras format)
# ─────────────────────────────────────────────────────────────────────────────

FINAL_MODEL_PATH = str(MODELS_DIR / f"{CONFIG['run_id']}.keras")
best_model.save(FINAL_MODEL_PATH)
print(f"Model saved → {FINAL_MODEL_PATH}")

Model saved → E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843.keras


In [47]:
# ─────────────────────────────────────────────────────────────────────────────
# F-2 | Save label_map.json
# Maps integer index → label string AND label string → integer index.
# Used by the inference module (Task 2) without re-importing training code.
# ─────────────────────────────────────────────────────────────────────────────

label_map = {
    "idx_to_label"  : {str(k): v for k, v in idx_to_label.items()},
    "label_to_idx"  : label_to_idx,
    "label_names"   : label_names,
    "num_classes"   : NUM_CLASSES,
    "backbone"      : CONFIG["backbone_name"],
    "image_size"    : list(CONFIG["image_size"]),
    "run_id"        : CONFIG["run_id"],
}

LABEL_MAP_PATH = str(RUN_DIR / "label_map.json")
with open(LABEL_MAP_PATH, "w") as f:
    json.dump(label_map, f, indent=2)
print(f"label_map.json saved → {LABEL_MAP_PATH}")

label_map.json saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\label_map.json


In [48]:
# ─────────────────────────────────────────────────────────────────────────────
# F-3 | Save metrics.json
# Serialise all scalar metrics plus per-class accuracy.
# Classification report text is saved separately for readability.
# ─────────────────────────────────────────────────────────────────────────────

# Separate text report from JSON-serialisable metrics
clf_report_text = eval_metrics.pop("classification_report")

METRICS_PATH = str(RUN_DIR / "metrics.json")
with open(METRICS_PATH, "w") as f:
    json.dump(eval_metrics, f, indent=2)
print(f"metrics.json saved → {METRICS_PATH}")

# Save classification report as plain text for human readability
CLF_REPORT_PATH = str(RUN_DIR / "classification_report.txt")
with open(CLF_REPORT_PATH, "w") as f:
    f.write(clf_report_text)
print(f"classification_report.txt saved → {CLF_REPORT_PATH}")

# Restore for inline display
eval_metrics["classification_report"] = clf_report_text

metrics.json saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\metrics.json
classification_report.txt saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\classification_report.txt


In [49]:
# ─────────────────────────────────────────────────────────────────────────────
# F-4 | Final Artifact Summary
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "═" * 65)
print(" TRAINING & EVALUATION COMPLETE")
print("═" * 65)
print(f" Run ID          : {CONFIG['run_id']}")
print(f" Backbone        : {CONFIG['backbone_name']}")
print(f" Test Accuracy   : {eval_metrics['overall_accuracy']:.4f}")
print(f" F1 (macro)      : {eval_metrics['f1_macro']:.4f}")
print(f" F1 (weighted)   : {eval_metrics['f1_weighted']:.4f}")
if eval_metrics.get("roc_auc_macro_ovr"):
    print(f" ROC-AUC (macro) : {eval_metrics['roc_auc_macro_ovr']:.4f}")
print("\n Saved artifacts:")
artifact_files = [
    FINAL_MODEL_PATH,
    CHECKPOINT_PATH,
    LABEL_MAP_PATH,
    METRICS_PATH,
    CLF_REPORT_PATH,
    CM_SAVE_PATH,
    MISCLASSIFIED_SAVE_PATH,
    ROC_SAVE_PATH,
    HISTORY_CSV_PATH,
    HISTORY_PLOT_PATH,
]
for path in artifact_files:
    exists = "✓" if pathlib.Path(path).exists() else "✗ MISSING"
    print(f"   {exists}  {path}")
print("═" * 65)


═════════════════════════════════════════════════════════════════
 TRAINING & EVALUATION COMPLETE
═════════════════════════════════════════════════════════════════
 Run ID          : 20260226_141843
 Backbone        : EfficientNetB0
 Test Accuracy   : 0.9823
 F1 (macro)      : 0.9834
 F1 (weighted)   : 0.9823
 ROC-AUC (macro) : 0.9999

 Saved artifacts:
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843.keras
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\20260226_141843_best.keras
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\label_map.json
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\metrics.json
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\classification_report.txt
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\confusion_matrix.png
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\misclassified_grid.png
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\2026

---

## Section G

> **Cells are self-contained** — they load the best model and `label_map.json`
> that were saved in Task 1 and do **not** re-run any training.
>
> **Sub-sections:**
> - G-1  Image bytes ↔ tensor adapter utility (`decode_image_bytes_to_tensor`)
> - G-2  `predict_image()` – accepts file path **or** raw bytes
> - G-3  `MinioLoader` stub (extended with full method signatures for Task 2)
> - G-4  Save `inference.py` module to `src/agritwin_gh/models/`
> - G-5  Save `deployment_notes.txt` to the artifacts folder
> - G-6  Sanity tests (path mode vs bytes mode)

In [50]:
# ─────────────────────────────────────────────────────────────────────────────
# G-1 | Adapter Utility — decode_image_bytes_to_tensor
#
# PURPOSE: Bridge function so that MinIO bytes (Task 2) can be decoded into a
#          preprocessed tf.Tensor using the EXACT same pipeline as training.
#          predict_image() calls this internally; the adapter is also exported
#          into inference.py so the MinIO integration (Task 2 server-side)
#          can import it without duplicating preprocessing logic.
# ─────────────────────────────────────────────────────────────────────────────

def decode_image_bytes_to_tensor(
    image_bytes    : bytes,
    image_size     : Tuple[int, int],
    preprocess_fn,
) -> tf.Tensor:
    """
    Decode raw image bytes into a preprocessed, batch-ready tf.Tensor.

    This function is the single entry-point for byte-based image input so
    that MinIO-fetched bytes can be fed into the model without core rewrites.

    Args:
        image_bytes  : Raw bytes of any supported image format (JPEG, PNG, …).
        image_size   : (H, W) target spatial size — must match training config.
        preprocess_fn: Backbone-specific normalisation function
                       (e.g. keras.applications.efficientnet.preprocess_input).

    Returns:
        tf.Tensor of shape (1, H, W, 3), dtype float32 — ready for model.predict().
    """
    H, W = image_size
    # Decode → float32 [H, W, 3]
    image = tf.image.decode_image(
        tf.constant(image_bytes),
        channels          = 3,
        expand_animations = False,
    )
    image = tf.image.resize(image, [H, W])
    image = tf.cast(image, tf.float32)
    # Apply backbone normalisation (same function used during training)
    image = preprocess_fn(image)
    # Add batch dimension → (1, H, W, 3)
    image = tf.expand_dims(image, axis=0)
    return image


print("decode_image_bytes_to_tensor — defined.")

decode_image_bytes_to_tensor — defined.


In [51]:
# ─────────────────────────────────────────────────────────────────────────────
# G-2 | predict_image() — Deployment-Ready Inference Function
#
# Accepts EITHER a local file path (str / pathlib.Path) OR raw bytes.
# Uses the label_map.json + backbone preprocess that were saved in Task 1.
# Returns a structured dict usable by a REST API or CLI without changes.
# ─────────────────────────────────────────────────────────────────────────────

from typing import Union

def predict_image(
    image_input     : Union[str, bytes, pathlib.Path],
    inference_model : keras.Model,
    loaded_label_map: dict,
    top_k           : int = 3,
) -> dict:
    """
    Run inference on a single image and return a structured prediction dict.

    Args:
        image_input      : One of:
                           (a) str or pathlib.Path — absolute / relative file path.
                           (b) bytes               — raw image bytes
                                                     (e.g. fetched from MinIO).
        inference_model  : Loaded Keras model (output: softmax probabilities).
        loaded_label_map : Dict loaded from label_map.json (Task 1 artifact).
                           Must contain keys: 'idx_to_label', 'image_size',
                           'backbone', 'label_names'.
        top_k            : Number of top-ranked classes to include in 'topk'.

    Returns:
        {
            "class_name" : str            — predicted class label,
            "confidence" : float          — probability of predicted class [0, 1],
            "probs"      : {label: float} — full probability dict over all classes,
            "topk"       : [(label, float), ...]  — top-k (label, prob) pairs,
            "input_mode" : "path" | "bytes",      — which input mode was used,
        }

    Raises:
        ValueError : if image_input is not a recognised type.
        FileNotFoundError: if a path is given but does not exist.
    """
    # ── Resolve backbone preprocess function from stored backbone name ────────
    _preprocess_map = {
        "EfficientNetB0"   : keras.applications.efficientnet.preprocess_input,
        "EfficientNetB3"   : keras.applications.efficientnet.preprocess_input,
        "ResNet50"         : keras.applications.resnet.preprocess_input,
        "MobileNetV3Large" : keras.applications.mobilenet_v3.preprocess_input,
        "DenseNet121"      : keras.applications.densenet.preprocess_input,
    }
    backbone_name   = loaded_label_map["backbone"]
    preprocess_fn   = _preprocess_map.get(backbone_name)
    if preprocess_fn is None:
        raise ValueError(
            f"Backbone '{backbone_name}' not found in inference preprocess map. "
            f"Add its preprocess_input to _preprocess_map inside predict_image()."
        )

    img_size_hw     = tuple(loaded_label_map["image_size"])   # (H, W)
    idx_to_label    = {int(k): v for k, v in loaded_label_map["idx_to_label"].items()}
    label_names     = loaded_label_map["label_names"]

    # ── Decode input to a preprocessed batch tensor ───────────────────────────
    input_mode: str

    if isinstance(image_input, (str, pathlib.Path)):
        # (a) File path mode
        fpath = pathlib.Path(image_input)
        if not fpath.exists():
            raise FileNotFoundError(f"Image file not found: {fpath}")
        image_bytes = fpath.read_bytes()
        input_mode  = "path"

    elif isinstance(image_input, (bytes, bytearray)):
        # (b) Raw bytes mode — ready for MinIO integration
        image_bytes = bytes(image_input)
        input_mode  = "bytes"

    else:
        raise ValueError(
            f"image_input must be str, pathlib.Path, or bytes. "
            f"Got: {type(image_input).__name__}"
        )

    # Shared decoding path — both modes converge here
    input_tensor = decode_image_bytes_to_tensor(
        image_bytes  = image_bytes,
        image_size   = img_size_hw,
        preprocess_fn= preprocess_fn,
    )   # shape: (1, H, W, 3)

    # ── Run inference ─────────────────────────────────────────────────────────
    prob_vector = inference_model.predict(input_tensor, verbose=0)[0]  # (num_classes,)
    prob_vector = prob_vector.astype(float)

    # ── Build output dict ─────────────────────────────────────────────────────
    pred_idx    = int(np.argmax(prob_vector))
    class_name  = idx_to_label[pred_idx]
    confidence  = float(prob_vector[pred_idx])

    probs_dict  = {idx_to_label[i]: float(prob_vector[i]) for i in range(len(prob_vector))}

    # Top-K classes sorted by probability (descending)
    top_k_pairs = sorted(probs_dict.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

    return {
        "class_name" : class_name,
        "confidence" : confidence,
        "probs"      : probs_dict,
        "topk"       : top_k_pairs,
        "input_mode" : input_mode,
    }


print("predict_image() — defined.")

predict_image() — defined.


In [52]:
# ─────────────────────────────────────────────────────────────────────────────
# G-3 | MinioLoader — EXTENDED STUB (Task 2 full method surface)
#
# Replaces the basic stub from Task 1 with the complete set of method signatures
# needed for a production MinIO-backed training pipeline.
# ALL methods raise NotImplementedError — implement in server-side Task 2.
#
# Integration contract:
#   • build_dataset() returns a tf.data.Dataset with the SAME signature as
#     pairs_to_tf_dataset() from Task 1, so the training loop is unchanged.
#   • fetch_image_bytes() returns raw bytes, which are passed directly to
#     decode_image_bytes_to_tensor() → no changes to preprocessing.
# ─────────────────────────────────────────────────────────────────────────────

class MinioLoader(DatasetLoader):
    """
    (EXTENDED STUB — Task 2) Full method surface for a MinIO-backed image loader.

    Inherits from DatasetLoader to satisfy the same interface used by
    LocalFolderLoader, ensuring the training loop requires zero changes.

    Args:
        minio_endpoint    : MinIO server URL, e.g. 'localhost:9000'.
        bucket_name       : MinIO bucket containing the image objects.
        label_prefix_map  : Dict  { object_key_prefix: label_string }
                            e.g. {'diseases/early_blight/': 'tomato_early_blight'}
        access_key        : MinIO access key (use env var in production).
        secret_key        : MinIO secret key (use env var in production).
        local_cache_dir   : Local directory to cache downloaded image bytes.
        secure            : Whether to use TLS (False for local dev).
    """

    def __init__(
        self,
        minio_endpoint  : str,
        bucket_name     : str,
        label_prefix_map: Dict[str, str],
        access_key      : str,
        secret_key      : str,
        local_cache_dir : str  = "/tmp/minio_image_cache",
        secure          : bool = False,
    ) -> None:
        # TODO (Task 2):
        #   from minio import Minio
        #   self._client = Minio(minio_endpoint, access_key=access_key,
        #                        secret_key=secret_key, secure=secure)
        #   self.bucket_name       = bucket_name
        #   self.label_prefix_map  = label_prefix_map
        #   self.local_cache_dir   = pathlib.Path(local_cache_dir)
        #   self.local_cache_dir.mkdir(parents=True, exist_ok=True)
        raise NotImplementedError(
            "MinioLoader.__init__ is a stub — implement in Task 2 server module."
        )

    # ── DatasetLoader interface ───────────────────────────────────────────────

    def load_file_label_pairs(self) -> List[Tuple[str, str]]:
        """
        (STUB) Download all objects for each prefix to local_cache_dir and
        return (local_cached_path, label) pairs — identical shape to
        LocalFolderLoader.load_file_label_pairs().

        Task 2 implementation outline:
            pairs = []
            for prefix, label in self.label_prefix_map.items():
                pairs.extend(self._build_local_cache(prefix, label))
            return pairs
        """
        raise NotImplementedError("MinioLoader.load_file_label_pairs — Task 2 stub.")

    def get_label_names(self) -> List[str]:
        """
        (STUB) Return sorted unique labels from label_prefix_map values.

        Task 2: return sorted(set(self.label_prefix_map.values()))
        """
        raise NotImplementedError("MinioLoader.get_label_names — Task 2 stub.")

    def describe(self) -> str:
        """
        (STUB) Return human-readable description including bucket and prefix counts.
        """
        raise NotImplementedError("MinioLoader.describe — Task 2 stub.")

    # ── Object listing & fetching ─────────────────────────────────────────────

    def list_objects(self, prefix: str) -> List[str]:
        """
        (STUB) List all object keys under `prefix` in the configured bucket.

        Task 2 implementation:
            objects = self._client.list_objects(
                self.bucket_name, prefix=prefix, recursive=True
            )
            return [obj.object_name for obj in objects]

        Args:
            prefix : Object key prefix to list, e.g. 'diseases/early_blight/'.

        Returns:
            List of full object key strings.
        """
        raise NotImplementedError("MinioLoader.list_objects — Task 2 stub.")

    def fetch_image_bytes(self, object_key: str) -> bytes:
        """
        (STUB) Download a single object and return its raw bytes.

        The returned bytes can be passed DIRECTLY to decode_image_bytes_to_tensor()
        or to predict_image() — no extra conversion needed.

        Task 2 implementation:
            response = self._client.get_object(self.bucket_name, object_key)
            data = response.read()
            response.close()
            response.release_conn()
            return data

        Args:
            object_key : Full MinIO object key, e.g. 'diseases/early_blight/img001.jpg'.

        Returns:
            Raw image bytes (JPEG / PNG / …).
        """
        raise NotImplementedError("MinioLoader.fetch_image_bytes — Task 2 stub.")

    def fetch_labels_map(self) -> Dict[str, str]:
        """
        (STUB) Optionally retrieve a remote label_map.json stored in MinIO
        (useful when the label mapping is managed centrally in the bucket).

        Task 2 implementation:
            raw = self.fetch_image_bytes('metadata/label_map.json')
            return json.loads(raw.decode('utf-8'))

        Returns:
            Dict equivalent to the label_map.json saved in Task 1.
        """
        raise NotImplementedError("MinioLoader.fetch_labels_map — Task 2 stub.")

    def build_dataset(
        self,
        label_to_idx   : Dict[str, int],
        image_size     : Tuple[int, int],
        preprocess_fn,
        batch_size     : int,
        shuffle        : bool = False,
        seed           : int  = 42,
    ) -> "tf.data.Dataset":
        """
        (STUB) Build a tf.data.Dataset that streams images from MinIO on-the-fly,
        avoiding the need to cache all images locally first.

        Task 2 implementation outline:
            1. For each prefix/label call list_objects(prefix)
            2. Build tf.data.Dataset.from_tensor_slices((object_keys, label_idxs))
            3. .map(lambda key, lbl: (
                   decode_image_bytes_to_tensor(
                       self.fetch_image_bytes(key.numpy().decode()),
                       image_size, preprocess_fn
                   )[0],
                   tf.one_hot(lbl, num_classes)
               ))
            4. .batch(batch_size).prefetch(tf.data.AUTOTUNE)

        Returns:
            tf.data.Dataset with (image_tensor, one_hot_label) pairs —
            IDENTICAL shape to the datasets built by pairs_to_tf_dataset()
            in Task 1, so the training loop is unchanged.
        """
        raise NotImplementedError("MinioLoader.build_dataset — Task 2 stub.")

    # ── Internal helpers ──────────────────────────────────────────────────────

    def _build_local_cache(
        self,
        prefix : str,
        label  : str,
    ) -> List[Tuple[str, str]]:
        """
        (STUB) List objects under `prefix`, download each to local_cache_dir,
        and return (local_path, label) pairs.

        Task 2: calls list_objects() then _download_object() per key.
        """
        raise NotImplementedError("MinioLoader._build_local_cache — Task 2 stub.")

    def _download_object(
        self,
        object_key : str,
        local_path : str,
    ) -> None:
        """
        (STUB) Download a single MinIO object to a local file path.

        Task 2: self._client.fget_object(self.bucket_name, object_key, local_path)
        """
        raise NotImplementedError("MinioLoader._download_object — Task 2 stub.")


print("MinioLoader (extended stub) — defined.")

MinioLoader (extended stub) — defined.


In [53]:
# ─────────────────────────────────────────────────────────────────────────────
# G-4 | Save inference.py module to src/agritwin_gh/models/
#
# This cell writes a standalone Python module that:
#   • Can be imported by a Flask/FastAPI service or CLI without Jupyter.
#   • Contains decode_image_bytes_to_tensor() and predict_image() verbatim.
#   • Reads label_map.json and loads the .keras model via public paths only.
#
# The module is intentionally self-contained (no imports from this notebook).
# ─────────────────────────────────────────────────────────────────────────────

INFERENCE_MODULE_SRC = '''"""
inference.py — AgriTwin-GH Tomato Disease Classifier
Deployment-ready inference module. Generated by Task-2 notebook.

Usage (standalone):
    from src.agritwin_gh.models.inference import load_inference_assets, predict_image

    model, label_map = load_inference_assets(
        model_path     = "src/agritwin_gh/models/<run_id>.keras",
        label_map_path = "src/agritwin_gh/models/artifacts/<run_id>/label_map.json",
    )
    result = predict_image(image_input="/path/to/leaf.jpg",
                           inference_model=model,
                           loaded_label_map=label_map)
    print(result)
"""

from __future__ import annotations

import json
import pathlib
from typing import Dict, List, Tuple, Union

import numpy as np
import tensorflow as tf
from tensorflow import keras


# ── Supported backbone → preprocess_input mapping ────────────────────────────
_BACKBONE_PREPROCESS_MAP: Dict[str, object] = {
    "EfficientNetB0"   : keras.applications.efficientnet.preprocess_input,
    "EfficientNetB3"   : keras.applications.efficientnet.preprocess_input,
    "ResNet50"         : keras.applications.resnet.preprocess_input,
    "MobileNetV3Large" : keras.applications.mobilenet_v3.preprocess_input,
    "DenseNet121"      : keras.applications.densenet.preprocess_input,
}


# ─────────────────────────────────────────────────────────────────────────────
# Public API
# ─────────────────────────────────────────────────────────────────────────────

def load_inference_assets(
    model_path     : Union[str, pathlib.Path],
    label_map_path : Union[str, pathlib.Path],
) -> Tuple[keras.Model, dict]:
    """
    Load a saved Keras model and the corresponding label_map.json.

    Args:
        model_path     : Path to the .keras (or SavedModel) file.
        label_map_path : Path to label_map.json saved by Task-1 training.

    Returns:
        (model, label_map_dict)
    """
    model = keras.models.load_model(str(model_path))
    with open(str(label_map_path), "r") as f:
        label_map = json.load(f)
    return model, label_map


def decode_image_bytes_to_tensor(
    image_bytes   : bytes,
    image_size    : Tuple[int, int],
    preprocess_fn,
) -> tf.Tensor:
    """
    Decode raw image bytes into a preprocessed, batch-ready tf.Tensor.

    This is the SINGLE decoding entry-point for byte-based image input so
    that MinIO-fetched bytes can be fed into the model without core rewrites.

    Args:
        image_bytes  : Raw bytes of any supported image format (JPEG, PNG, …).
        image_size   : (H, W) — must match training CONFIG image_size.
        preprocess_fn: Backbone-specific normalisation callable.

    Returns:
        tf.Tensor of shape (1, H, W, 3), dtype float32.
    """
    H, W = image_size
    image = tf.image.decode_image(
        tf.constant(image_bytes),
        channels          = 3,
        expand_animations = False,
    )
    image = tf.image.resize(image, [H, W])
    image = tf.cast(image, tf.float32)
    image = preprocess_fn(image)
    image = tf.expand_dims(image, axis=0)   # add batch dim
    return image


def predict_image(
    image_input     : Union[str, bytes, pathlib.Path],
    inference_model : keras.Model,
    loaded_label_map: dict,
    top_k           : int = 3,
) -> dict:
    """
    Run inference on a single image.

    Args:
        image_input      : File path (str / Path) OR raw bytes.
        inference_model  : Loaded Keras model.
        loaded_label_map : Dict from label_map.json.
        top_k            : Number of top classes to return.

    Returns:
        {
            "class_name" : str,
            "confidence" : float,
            "probs"      : {label: float, ...},
            "topk"       : [(label, float), ...],
            "input_mode" : "path" | "bytes",
        }
    """
    backbone_name = loaded_label_map["backbone"]
    preprocess_fn = _BACKBONE_PREPROCESS_MAP.get(backbone_name)
    if preprocess_fn is None:
        raise ValueError(
            f"Backbone \'{backbone_name}\' not in _BACKBONE_PREPROCESS_MAP."
        )

    img_size_hw  = tuple(loaded_label_map["image_size"])
    idx_to_label = {int(k): v for k, v in loaded_label_map["idx_to_label"].items()}

    # -- Input resolution ---
    if isinstance(image_input, (str, pathlib.Path)):
        fpath = pathlib.Path(image_input)
        if not fpath.exists():
            raise FileNotFoundError(f"Image file not found: {fpath}")
        image_bytes = fpath.read_bytes()
        input_mode  = "path"
    elif isinstance(image_input, (bytes, bytearray)):
        image_bytes = bytes(image_input)
        input_mode  = "bytes"
    else:
        raise ValueError(
            f"image_input must be str, pathlib.Path, or bytes. "
            f"Got: {type(image_input).__name__}"
        )

    input_tensor = decode_image_bytes_to_tensor(image_bytes, img_size_hw, preprocess_fn)
    prob_vector  = inference_model.predict(input_tensor, verbose=0)[0].astype(float)

    pred_idx    = int(np.argmax(prob_vector))
    class_name  = idx_to_label[pred_idx]
    confidence  = float(prob_vector[pred_idx])
    probs_dict  = {idx_to_label[i]: float(prob_vector[i]) for i in range(len(prob_vector))}
    top_k_pairs = sorted(probs_dict.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

    return {
        "class_name" : class_name,
        "confidence" : confidence,
        "probs"      : probs_dict,
        "topk"       : top_k_pairs,
        "input_mode" : input_mode,
    }
'''

# ── Write the module ──────────────────────────────────────────────────────────
INFERENCE_PY_PATH = REPO_ROOT / "src" / "agritwin_gh" / "models" / "inference.py"
INFERENCE_PY_PATH.parent.mkdir(parents=True, exist_ok=True)
INFERENCE_PY_PATH.write_text(INFERENCE_MODULE_SRC, encoding="utf-8")
print(f"inference.py saved → {INFERENCE_PY_PATH}")

inference.py saved → E:\AgriTwin-GH\src\agritwin_gh\models\inference.py


In [54]:
# ─────────────────────────────────────────────────────────────────────────────
# G-5 | Save deployment_notes.txt into the run artifacts folder
# ─────────────────────────────────────────────────────────────────────────────

DEPLOYMENT_NOTES = f"""AgriTwin-GH — Tomato Disease Classifier
Deployment Notes
Run ID  : {CONFIG['run_id']}
Backbone: {CONFIG['backbone_name']}
Generated: {datetime.datetime.now().isoformat(timespec='seconds')}
========================================================

MODEL ARTIFACTS
---------------
Model file  : src/agritwin_gh/models/{CONFIG['run_id']}.keras
Best ckpt   : src/agritwin_gh/models/{CONFIG['run_id']}_best.keras
label_map   : src/agritwin_gh/models/artifacts/{CONFIG['run_id']}/label_map.json
metrics     : src/agritwin_gh/models/artifacts/{CONFIG['run_id']}/metrics.json
Inference   : src/agritwin_gh/models/inference.py


LOADING THE MODEL
-----------------
    from src.agritwin_gh.models.inference import load_inference_assets, predict_image

    model, label_map = load_inference_assets(
        model_path     = "src/agritwin_gh/models/{CONFIG['run_id']}.keras",
        label_map_path = "src/agritwin_gh/models/artifacts/{CONFIG['run_id']}/label_map.json",
    )


RUNNING INFERENCE
-----------------
# (a) From a file path:
    result = predict_image(
        image_input      = "/path/to/leaf.jpg",
        inference_model  = model,
        loaded_label_map = label_map,
        top_k            = 3,
    )

# (b) From raw bytes (MinIO-compatible):
    image_bytes = open("/path/to/leaf.jpg", "rb").read()
    # — or — image_bytes = minio_client.get_object(bucket, key).read()
    result = predict_image(
        image_input      = image_bytes,
        inference_model  = model,
        loaded_label_map = label_map,
    )

# Result structure:
    {{
        "class_name" : "tomato_early_blight",
        "confidence" : 0.9721,
        "probs"      : {{ "tomato_early_blight": 0.9721, ... }},
        "topk"       : [("tomato_early_blight", 0.9721), ...],
        "input_mode" : "path"  # or "bytes"
    }}


IMAGE PREPROCESSING (must match training)
------------------------------------------
  image_size : {CONFIG['image_size']}   (H × W)
  backbone   : {CONFIG['backbone_name']}
  normaliser : keras.applications.{CONFIG['backbone_name'].lower()}.preprocess_input
  (handled automatically by predict_image / decode_image_bytes_to_tensor)


CLASSES ({CONFIG['num_classes']} total)
-----------------------------------------
  0: tomato_early_blight
  1: tomato_late_blight
  2: tomato_leaf_healthy
  3: tomato_leaf_mold
  4: tomato_powdery_mildew
  5: tomato_spider_mites
  (verify order against label_map.json — it is the authoritative source)


MINIO INTEGRATION (Task 2 — stub ready)
-----------------------------------------
MinioLoader is defined in the notebook. To enable:
  1. pip install minio
  2. Implement the methods marked NotImplementedError in MinioLoader.
  3. In CONFIG set: "loader": "minio"
  4. Provide minio_endpoint, bucket_name, label_prefix_map, credentials.
  5. fetch_image_bytes(key) returns bytes → pass directly to predict_image().
"""

DEPLOYMENT_NOTES_PATH = RUN_DIR / "deployment_notes.txt"
DEPLOYMENT_NOTES_PATH.write_text(DEPLOYMENT_NOTES, encoding="utf-8")
print(f"deployment_notes.txt saved → {DEPLOYMENT_NOTES_PATH}")

deployment_notes.txt saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\deployment_notes.txt


In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# G-6 | Sanity Tests — Load Saved Artifacts & Run Both Inference Modes
#
# Tests performed (no retraining):
#   T1 — Load best_model from .keras checkpoint  ✓
#   T2 — Load label_map.json                     ✓
#   T3 — predict_image() with a file path        ✓
#   T4 — predict_image() with raw bytes          ✓
#   T5 — Path-mode and bytes-mode probs match    ✓
#   T6 — Top-3 printed for human inspection      ✓
# ─────────────────────────────────────────────────────────────────────────────

import pprint

# ── T1 + T2: Load artifacts ───────────────────────────────────────────────────
print("T1 — Loading model from checkpoint ...")
_sanity_model = keras.models.load_model(CHECKPOINT_PATH)
print(f"     Model loaded: {_sanity_model.name}  "
      f"(input: {_sanity_model.input_shape}, output: {_sanity_model.output_shape})")

print("\nT2 — Loading label_map.json ...")
with open(LABEL_MAP_PATH, "r") as _f:
    _sanity_label_map = json.load(_f)
print(f"     Classes: {_sanity_label_map['label_names']}")

# ── Pick a random test image path for both tests ──────────────────────────────
_rng_sample           = random.Random(CONFIG["random_seed"] + 99)
_sample_path_str, _sample_true_label = _rng_sample.choice(test_pairs)
_sample_path          = pathlib.Path(_sample_path_str)

print(f"\nSample image  : {_sample_path.name}")
print(f"True label    : {_sample_true_label}")

# ── T3: File-path mode ────────────────────────────────────────────────────────
print("\nT3 — predict_image() [path mode] ...")
_result_path = predict_image(
    image_input      = _sample_path,
    inference_model  = _sanity_model,
    loaded_label_map = _sanity_label_map,
    top_k            = 3,
)
print(f"     input_mode : {_result_path['input_mode']}")
print(f"     class_name : {_result_path['class_name']}")
print(f"     confidence : {_result_path['confidence']:.4f}")
print(f"     top-3      :")
for rank, (cls, prob) in enumerate(_result_path["topk"], start=1):
    print(f"       {rank}. {cls:<35}  {prob:.4f}")

# ── T4: Bytes mode ────────────────────────────────────────────────────────────
print("\nT4 — predict_image() [bytes mode] ...")
_sample_bytes  = _sample_path.read_bytes()
_result_bytes  = predict_image(
    image_input      = _sample_bytes,
    inference_model  = _sanity_model,
    loaded_label_map = _sanity_label_map,
    top_k            = 3,
)
print(f"     input_mode : {_result_bytes['input_mode']}")
print(f"     class_name : {_result_bytes['class_name']}")
print(f"     confidence : {_result_bytes['confidence']:.4f}")
print(f"     top-3      :")
for rank, (cls, prob) in enumerate(_result_bytes["topk"], start=1):
    print(f"       {rank}. {cls:<35}  {prob:.4f}")

# ── T5: Path-mode and bytes-mode probability vectors must be identical ─────────
print("\nT5 — Comparing path-mode vs bytes-mode probability vectors ...")
_prob_path_arr  = np.array([_result_path["probs"][lbl]  for lbl in _sanity_label_map["label_names"]])
_prob_bytes_arr = np.array([_result_bytes["probs"][lbl] for lbl in _sanity_label_map["label_names"]])
_max_diff       = float(np.max(np.abs(_prob_path_arr - _prob_bytes_arr)))
print(f"     Max absolute probability diff: {_max_diff:.2e}  (expect < 1e-5)")
assert _max_diff < 1e-5, (
    f"FAIL: path-mode and bytes-mode probabilities diverged by {_max_diff:.2e}"
)
print("     ✓  Both inference modes produce identical outputs.")

# ── T6: Correctness check (soft — just log; don't fail if wrong) ──────────────
print(f"\nT6 — Correctness check (soft):")
print(f"     True label : {_sample_true_label}")
print(f"     Prediction : {_result_path['class_name']}")
if _result_path["class_name"] == _sample_true_label:
    print("     ✓  Correct prediction.")
else:
    print("     ✗  Incorrect prediction (acceptable on a single sample).")

print("\n" + "═" * 60)
print(" ALL SANITY TESTS PASSED — inference module is deployment-ready.")
print("═" * 60)

T1 — Loading model from checkpoint ...
     Model loaded: EfficientNetB0_classifier  (input: (None, 224, 224, 3), output: (None, 6))

T2 — Loading label_map.json ...
     Classes: ['tomato_early_blight', 'tomato_late_blight', 'tomato_leaf_healthy', 'tomato_leaf_mold', 'tomato_powdery_mildew', 'tomato_spider_mites']

Sample image  : pm151_change_180.jpg
True label    : tomato_powdery_mildew

T3 — predict_image() [path mode] ...
     input_mode : path
     class_name : tomato_powdery_mildew
     confidence : 0.7754
     top-3      :
       1. tomato_powdery_mildew                0.7754
       2. tomato_late_blight                   0.0606
       3. tomato_leaf_mold                     0.0552

T4 — predict_image() [bytes mode] ...
     input_mode : bytes
     class_name : tomato_powdery_mildew
     confidence : 0.7754
     top-3      :
       1. tomato_powdery_mildew                0.7754
       2. tomato_late_blight                   0.0606
       3. tomato_leaf_mold                     

In [56]:
# ─────────────────────────────────────────────────────────────────────────────
# G-7 | Task-2 Artifact Summary
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "═" * 65)
print(" TASK 2 COMPLETE — DEPLOYMENT ARTIFACTS")
print("═" * 65)

task2_artifacts = {
    "inference.py (module)"   : INFERENCE_PY_PATH,
    "deployment_notes.txt"    : DEPLOYMENT_NOTES_PATH,
}

for desc, path in task2_artifacts.items():
    exists = "✓" if pathlib.Path(path).exists() else "✗ MISSING"
    print(f"   {exists}  {path}")

print("\n Pipeline modularity summary:")
print("   LocalFolderLoader  → FULLY IMPLEMENTED (Task 1)")
print("   MinioLoader        → STUB with full method surface (ready for Task 2 server)")
print("   decode_image_bytes_to_tensor → exported to inference.py")
print("   predict_image()    → exported to inference.py")
print("\n   To switch data source:  set CONFIG['loader'] = 'minio'")
print("   Training loop, augmentation, model, callbacks — NO CHANGES NEEDED.")
print("═" * 65)


═════════════════════════════════════════════════════════════════
 TASK 2 COMPLETE — DEPLOYMENT ARTIFACTS
═════════════════════════════════════════════════════════════════
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\inference.py
   ✓  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\20260226_141843\deployment_notes.txt

 Pipeline modularity summary:
   LocalFolderLoader  → FULLY IMPLEMENTED (Task 1)
   MinioLoader        → STUB with full method surface (ready for Task 2 server)
   decode_image_bytes_to_tensor → exported to inference.py
   predict_image()    → exported to inference.py

   To switch data source:  set CONFIG['loader'] = 'minio'
   Training loop, augmentation, model, callbacks — NO CHANGES NEEDED.
═════════════════════════════════════════════════════════════════
